# Multimodal Retrieval Demo
This notebook showcases the procedurally generated几何图形 dataset and the multimodal retrieval pipeline.

In [ ]:
from pathlib import Path
import json
import time
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from dierpian.data.generate_shapes_dataset import generate_shapes_dataset
from dierpian.graphs.pmi_attribute_graph import PMIAttributeGraph
from dierpian.retrieval.multimodal_retriever import MultiModalRetriever


In [ ]:
DATA_DIR = Path("data")
metadata_path = DATA_DIR / "metadata.json"
if not metadata_path.exists():
    print("Generating dataset...")
    generate_shapes_dataset(DATA_DIR)

metadata = json.loads(metadata_path.read_text())
df = pd.DataFrame(metadata)
display(df.head())

In [ ]:
graph_builder = PMIAttributeGraph(min_pmi=0.0)
graph_builder.fit(metadata)
retriever = MultiModalRetriever(similarity="cosine", graph_weight=0.6, visual_weight=0.4)
retriever.fit(metadata, image_root=DATA_DIR)
print(f"Graph nodes: {retriever.graph.graph.number_of_nodes()}")
print(f"Dataset size: {len(metadata)} samples")

In [ ]:
query_index = 0
row = df.iloc[query_index]
print("Query attributes:", row.to_dict())
indices = retriever.query_by_index(query_index, top_k=6)
print("Retrieved indices:", indices)
fig, axes = plt.subplots(1, len(indices), figsize=(3 * len(indices), 3))
for ax, idx in zip(axes, indices):
    sample = df.iloc[idx]
    ax.imshow(plt.imread(DATA_DIR / sample['filename']))
    ax.set_title(f"{sample['shape']}
{sample['color']}")
    ax.axis('off')
plt.show()

In [ ]:
results = retriever.evaluate(range(0, len(metadata), 5), top_k=5, attribute="shape")
print("Evaluation metrics:", results)

In [ ]:
log = pd.DataFrame([results])
log['timestamp'] = pd.Timestamp.utcnow()
display(log)